# GPU：复用 CPU 缓存训练
选择 L4/A100 GPU 运行时，依次执行。先完成 CPU notebook，再运行这里。
与 CPU 使用相同的数据目录选择、本地 `/content/studies_links` 软链接和 `/content/local_cache` 缓存路径。
原始数据路径、数据版本、滤波和采样参数必须与预热阶段一致。
`--require-cache` 禁止补算原始信号；缺失缓存会立即报错。
`--resume auto` 恢复同一目录的 last.ckpt；总轮数仍是 EPOCHS，不会额外再加 EPOCHS。
使用「全部运行」即可依次完成安装、缓存解压和训练；耗时步骤会实时显示日志与报错。
`SUBJECT_ARGS` 可选择全量、单人、固定多人名单或前 N 个有效受试者；实际名单保存为 `subjects.json`。


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

from google.colab import drive


def run_live(command):
    with subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                          text=True, bufsize=1) as process:
        for line in process.stdout:
            print(line, end="", flush=True)
        if process.wait():
            raise subprocess.CalledProcessError(process.returncode, command)


drive.mount("/content/drive")
REPO_URL = "https://github.com/wdongdong0708-jpg/brain2qwertyv1Test.git"
CODE = Path("/content/code")
WORKSPACE = Path("/content/drive/MyDrive/Brain2Qwerty_Workspace")
if (CODE / ".git").exists():
    subprocess.run(["git", "-C", str(CODE), "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", REPO_URL, str(CODE)], check=True)
os.chdir(CODE)
for folder in ("data", "archives", "runs"):
    (WORKSPACE / folder).mkdir(parents=True, exist_ok=True)
raw_candidates = [WORKSPACE / "data/SpanishBCBL", WORKSPACE / "data/SpanishBCBL_Full"]
raw = next((path for path in raw_candidates if path.is_dir()), None)
if raw is None:
    raise FileNotFoundError(f"SpanishBCBL data directory not found; checked: {raw_candidates}")
raw = raw.resolve()
STUDIES_LINKS = Path("/content/studies_links")
STUDIES_LINKS.mkdir(parents=True, exist_ok=True)
for name in ("Pinet2024Eeg", "Pinet2024Meg"):
    alias = STUDIES_LINKS / name
    if alias.is_symlink() and alias.resolve() != raw:
        alias.unlink()
    if not alias.is_symlink():
        if alias.exists():
            raise FileExistsError(f"Expected a local symlink, but found an existing directory/file: {alias}")
        alias.symlink_to(raw, target_is_directory=True)
os.environ["BRAIN2QWERTY_STUDIES"] = str(STUDIES_LINKS)
os.environ["BRAIN2QWERTY_CACHE"] = "/content/local_cache"
Path(os.environ["BRAIN2QWERTY_CACHE"]).mkdir(parents=True, exist_ok=True)
os.environ["BRAIN2QWERTY_RESULTS"] = str(WORKSPACE / "runs")
print(f"Raw data: {raw}; study links: {STUDIES_LINKS}; cache: /content/local_cache")
subprocess.run(["git", "log", "-1", "--oneline"], check=True)


In [ ]:
run_live([sys.executable, "-u", "-m", "scripts.colab_setup"])
PYTHON = "/content/b2q-venv/bin/python"

run_live([PYTHON, "-c", "import torch, brain2qwerty_v1; print('GPU available:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'No GPU')"])


In [ ]:
os.environ["BRAIN2QWERTY_CACHE"] = "/content/local_cache"
for archive_path in (WORKSPACE / "archives/features_meg.tar", WORKSPACE / "archives/features.tar"):
    if archive_path.is_file():
        print(f"Unpacking {archive_path} -> {os.environ['BRAIN2QWERTY_CACHE']}...", flush=True)
        run_live([PYTHON, "-u", "-m", "scripts.cache_archive", "unpack",
                  "--archive", str(archive_path),
                  "--cache-root", os.environ["BRAIN2QWERTY_CACHE"]])


## 实验一 & 实验二：词级打字时长捷径验证 +《From Thought to Action》预运动超前窗与跨步解混
直接复用已解压的缓存，在 Colab 中自动下载/缓存 `BAAI/bge-m3` 西班牙语词向量，一次性运行：
- **第一步（修好 `Rolling-K0/K1/K2` 与词性掩码）**：零初始化门控残差 $\alpha_r(0)=0$ 严格锁死 `Word-Only` 下界 + 局部相对位置窗口 `[i-2, i+r]` 施密特正交解混 + 全词表分层抽样恢复西语 `Func / Content` 词性统计。
- **第二步（《From Thought to Action》预运动超前窗口与并发叠加子空间解混）**：在固定窗长 `T=0.50s` 下对比运动后窗 `[+0.00s, +0.50s]`、预运动跨期窗 `[-0.25s, +0.25s]` 与早期预运动窗 `[-0.35s, +0.15s]`，展示 `Rolling-K0` 等待延迟砍半（`0.50s -> 0.25s`）以及 `Word-Only -> K0 -> K1 -> K2` 的渐进解混跃升。
- **实验二（模拟 Brain2Qwerty v2 空格边界切分 `Space-Chunked [τ_i, τ_{i+1}]` 模式）**：对比独立空格切块均值池化（`Exp2-1`）、通道 A（`Exp2-2`）、通道 B（`Exp2-3`）、完整 v2 联合训练（`Exp2-4`）及去偏方案（`Exp2-5`）。
验证结果实时打印三张 Markdown 汇总表，并保存到 Drive 的 `runs/<tag>/shortcut_verification.{md,csv,json}`。


In [ ]:
subprocess.run(["git", "-C", str(CODE), "pull", "--ff-only"], check=True)
SHORTCUT_MODALITY = "meg"   # meg (features_meg.tar) / eeg (features.tar)
SHORTCUT_SUBJECTS = ["--num-subjects", "1"]  # 快速单人验证；3人填 ["3"]，全量填 ["--subject", "all"]
SHORTCUT_PROFILE = "small"  # 快速验证用 "small" (~1-2分钟跑完)；正式全量用 "full"
EXPERIMENTS = ["exp1", "exp2"]  # ["exp1", "exp2"] 同时跑实验一(含Step1+Step2)与实验二，或单独跑 ["exp1"]
WINDOW_DURATION = 0.5           # 固定词窗时长 T=0.50s
WINDOW_OFFSETS = ["0.0", "-0.25", "-0.35"]  # 对比 [+0.00s, +0.50s], [-0.25s, +0.25s], [-0.35s, +0.15s]
SHORTCUT_EPOCHS = 15
run_live([PYTHON, "-u", "-m", "brain2qwerty_v1.scripts.verify_word_shortcut",
          "--modality", SHORTCUT_MODALITY, *SHORTCUT_SUBJECTS,
          "--profile", SHORTCUT_PROFILE,
          "--experiments", *EXPERIMENTS,
          "--window-duration", str(WINDOW_DURATION),
          "--window-offsets", *WINDOW_OFFSETS,
          "--epochs", str(SHORTCUT_EPOCHS),
          "--require-cache"])


## 实验 B：原始 V1 键击级完整训练（可选）


In [ ]:
MODALITY = "eeg"   # eeg / meg
SUBJECT_ARGS = ["--subject", "all"]
# 10 participants: ["--num-subjects", "10"]
# Fixed list: ["--subjects", "002", "003", "004"]
# One participant: ["--subject", "002"] (EEG) or ["--subject", "S2"] (MEG)
EPOCHS = 300       # finite total budget; resume must keep the original value
SEED = 33
run_live([PYTHON, "-u", "-m", "brain2qwerty_v1", "train",
          "--modality", MODALITY, *SUBJECT_ARGS,
          "--seed", str(SEED), "--epochs", str(EPOCHS),
          "--devices", "1", "--require-cache", "--resume", "auto"])


Checkpoint、逐步日志、每轮状态均直接写入 Drive 的 runs/。同一实验再次执行会恢复断点；模型结构、batch、种子和总轮数需保持一致。新实验请指定不同的 `--run-dir`。
